# Parliament comparison

All comparison values come from the same analysis functions used by the focused notebooks.

In [1]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

import pandas as pd

root = Path(os.environ.get("APEMAP_PROJECT_ROOT", Path.cwd())).resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from apemap.analysis import (
    compute_funding_summary,
    compute_parliament_demographics,
    compute_sector_summary,
)
from apemap.db import get_connection

db_path = Path(
    os.environ.get("APEMAP_DB_PATH", root / "data" / "aped.duckdb")
).resolve()
conn = get_connection(db_path, read_only=True)

In [2]:
rows = []
for parliament in (46, 47, 48):
    demographics = compute_parliament_demographics(conn, parliament)
    sectors = compute_sector_summary(conn, parliament)
    finance = compute_funding_summary(conn, parliament)
    rows.append(
        {
            "parliament": parliament,
            "opening_date": demographics["reference_opening_date"],
            "parliamentarians": demographics["total_parliamentarians"],
            "known_age_n": demographics["known_age_sample_size"],
            "missing_age_n": demographics["missing_age_count"],
            "known_school_n": sectors["parliamentarians_with_known_schools"],
            "missing_school_n": sectors["parliamentarians_without_known_schools"],
            "finance_gross_n": finance["overall_gross_income"]["n"],
            "finance_gross_missing": finance["overall_gross_income"]["missing"],
        }
    )
pd.DataFrame(rows)

,parliament,opening_date,parliamentarians,known_age_n,missing_age_n,known_school_n,missing_school_n,finance_gross_n,finance_gross_missing
0,46,2019-07-02,229,222,7,198,40,126,186
1,47,2022-07-26,228,215,13,200,37,121,193
2,48,2025-07-22,226,194,32,181,49,101,191


In [3]:
conn.close()